In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
import glob
import os
import pandas as pd
import multiprocessing
import logging
import blimpy as bl
%matplotlib inline

In [2]:
df = pd.read_csv('/datax/scratch/benjb/bl_nearby_stars/BL_cadences_unique_nearby_star_sample_only.csv')
df.insert(0, column='Index', value=np.arange(len(df)))

unspliced_nums = []
for i, h5 in enumerate(df['.h5 path 1']):
    if not 'spliced' in h5:
        unspliced_nums.append(i)

pool_dats = np.sort(glob.glob('/datax/scratch/benjb/bl_nearby_stars/bliss_dats_pooled_090825/*.dat'))
skip_nums = []
for dat in pool_dats:
    n = int(os.path.basename(dat).split('_')[0])
    skip_nums.append(n)
skip_nums = np.unique(skip_nums)
skip_nums = np.concatenate([unspliced_nums, skip_nums])
skip_nums = unspliced_nums
skip_nums = np.sort(skip_nums)

skip_idx = [0, 1, 2, 63, 64, 65, 95, 96, 97, 121, 122, 123, 147, 148, 149,
             156, 157, 171, 172, 9178, 9179, 9180, 9197, 9198, 9199, 9200, 9201, 9202, 9203, 9204,
             9205, 9206, 9207, 9208, 9209, 9210, 9241, 9242, 19474, 19475, 19499, 19571, 19578, 19579, 
             19580, 19581, 19582, 19583, 19584, 19585, 28806, 28807, 28831, 28832, 28856, 28857, 28858, 
             28882, 28883, 28884, 28885, 28886, 28887, 28917, 28918, 28919, 28920, 28921, 28922, 28923]

skip_nums = np.concatenate([unspliced_nums, skip_idx])
skip_nums = np.sort(skip_nums)

print(len(df))

df.drop(index=skip_nums, inplace=True)

print(len(df))

39177
4468


In [18]:
list_to_move = []

In [19]:
# from blpc0 - take 72 files from each node

nnodes = 4
batch_size = len(df) // 3 // nnodes
for i in range(nnodes):
    file_set = df.iloc[i*batch_size+len(df)*0//4:(i+1)*batch_size+len(df)*0//4]
    rows_to_move = file_set[-72:]
    for j in range(6):
        list_to_move.append(rows_to_move[f'.h5 path {j+1}'].values)

In [20]:
# blpc1 appears to be almost done — take no files from this node

In [21]:
# from blpc2 - take 72 files from each node

nnodes = 4
batch_size = len(df) // 3 // nnodes
for i in range(nnodes):
    file_set = df.iloc[i*batch_size+len(df)*2//3:(i+1)*batch_size+len(df)*2//3]
    rows_to_move = file_set[-72:]
    for j in range(6):
        list_to_move.append(rows_to_move[f'.h5 path {j+1}'].values)
list_to_move = np.concatenate(list_to_move)
print(len(list_to_move))

3456


In [22]:
np.savetxt('/datax/scratch/benjb/bl_nearby_stars/list_of_files_for_gbo_move.txt', list_to_move, delimiter=',', fmt='%s')